In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('/content/drive/MyDrive/mlops/tweets.csv')

df.head()

,id,keyword,location,text,target
0,0,ablaze,NaN,"Communal violence in Bhainsa, Telangana. ""Ston...",1
1,1,ablaze,NaN,Telangana: Section 144 has been imposed in Bha...,1
2,2,ablaze,New York City,Arsonist sets cars ablaze at dealership https:...,1
3,3,ablaze,"Morgantown, WV",Arsonist sets cars ablaze at dealership https:...,1
4,4,ablaze,NaN,"""Lord Jesus, your love brings freedom and pard...",0


In [4]:
df.shape

(11370, 5)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11370 entries, 0 to 11369
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        11370 non-null  int64 
 1   keyword   11370 non-null  object
 2   location  7952 non-null   object
 3   text      11370 non-null  object
 4   target    11370 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 444.3+ KB


In [9]:
df.isnull().sum()

,0
id,0
keyword,0
location,3418
text,0
target,0


In [12]:
df = df[['keyword', 'text', 'target']].copy()

df.head()

,keyword,text,target
0,ablaze,"Communal violence in Bhainsa, Telangana. ""Ston...",1
1,ablaze,Telangana: Section 144 has been imposed in Bha...,1
2,ablaze,Arsonist sets cars ablaze at dealership https:...,1
3,ablaze,Arsonist sets cars ablaze at dealership https:...,1
4,ablaze,"""Lord Jesus, your love brings freedom and pard...",0


In [13]:
df['keyword'] = df['keyword'].fillna('')

df['text'] = df['text'].fillna('')

In [14]:
df.drop_duplicates(inplace=True)
df.shape

(11228, 3)

In [15]:
df['combined_text'] = df['keyword'] + " " + df['text']

df.head()

,keyword,text,target,combined_text
0,ablaze,"Communal violence in Bhainsa, Telangana. ""Ston...",1,"ablaze Communal violence in Bhainsa, Telangana..."
1,ablaze,Telangana: Section 144 has been imposed in Bha...,1,ablaze Telangana: Section 144 has been imposed...
2,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...
3,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...
4,ablaze,"""Lord Jesus, your love brings freedom and pard...",0,"ablaze ""Lord Jesus, your love brings freedom a..."


In [17]:
import re

def clean_text(text):

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+", "", text)

    # remove mentions
    text = re.sub(r"@\w+", "", text)

    # remove hashtags
    text = re.sub(r"#", "", text)

    # remove special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    return text

In [18]:
df['clean_text'] = df['combined_text'].apply(clean_text)

df.head()

,keyword,text,target,combined_text,clean_text
0,ablaze,"Communal violence in Bhainsa, Telangana. ""Ston...",1,"ablaze Communal violence in Bhainsa, Telangana...",ablaze communal violence in bhainsa telangana ...
1,ablaze,Telangana: Section 144 has been imposed in Bha...,1,ablaze Telangana: Section 144 has been imposed...,ablaze telangana section has been imposed in ...
2,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...,ablaze arsonist sets cars ablaze at dealership
3,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...,ablaze arsonist sets cars ablaze at dealership
4,ablaze,"""Lord Jesus, your love brings freedom and pard...",0,"ablaze ""Lord Jesus, your love brings freedom a...",ablaze lord jesus your love brings freedom and...


In [19]:
X = df['clean_text']

y = df['target']

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(X)

print(X.shape)

(11228, 21770)


In [22]:
df.head()

,keyword,text,target,combined_text,clean_text
0,ablaze,"Communal violence in Bhainsa, Telangana. ""Ston...",1,"ablaze Communal violence in Bhainsa, Telangana...",ablaze communal violence in bhainsa telangana ...
1,ablaze,Telangana: Section 144 has been imposed in Bha...,1,ablaze Telangana: Section 144 has been imposed...,ablaze telangana section has been imposed in ...
2,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...,ablaze arsonist sets cars ablaze at dealership
3,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...,ablaze arsonist sets cars ablaze at dealership
4,ablaze,"""Lord Jesus, your love brings freedom and pard...",0,"ablaze ""Lord Jesus, your love brings freedom a...",ablaze lord jesus your love brings freedom and...


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [24]:
from sklearn.linear_model import LogisticRegression

In [25]:
model = LogisticRegression()

In [26]:
model.fit(X_train,y_train)

LogisticRegression()

In [27]:
pred = model.predict(X_test)

print(pred[:10])

[1 0 0 0 0 0 0 0 0 0]


In [28]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred)

print("Accuracy:", accuracy)

Accuracy: 0.8753339269813001


In [29]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.88      0.99      0.93      1838
           1       0.88      0.37      0.52       408

    accuracy                           0.88      2246
   macro avg       0.88      0.68      0.72      2246
weighted avg       0.88      0.88      0.85      2246



In [30]:
sample = ["huge earthquake destroyed many buildings"]

sample_vector = tfidf.transform(sample)

result = model.predict(sample_vector)

print(result)

[1]


In [31]:
if result[0] == 1:
    print("Disaster Tweet")

else:
    print("Not Disaster Tweet")

Disaster Tweet


In [32]:
import pickle

with open("logistic_model.pkl", "wb") as file:
    pickle.dump(model, file)

with open("tfidf.pkl", "wb") as file:
    pickle.dump(tfidf, file)

print("Pickle files saved successfully")

Pickle files saved successfully


In [33]:
with open("logistic_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

with open("tfidf.pkl", "rb") as file:
    loaded_tfidf = pickle.load(file)

sample = ["flood destroyed roads"]

sample_vector = loaded_tfidf.transform(sample)

prediction = loaded_model.predict(sample_vector)

print(prediction)

[0]


In [34]:
import joblib

joblib.dump(model, "logistic_model.joblib")

joblib.dump(tfidf, "tfidf.joblib")

print("Joblib files saved successfully")

Joblib files saved successfully


In [35]:
loaded_model = joblib.load("logistic_model.joblib")

loaded_tfidf = joblib.load("tfidf.joblib")

sample = ["forest fire near city"]

sample_vector = loaded_tfidf.transform(sample)

prediction = loaded_model.predict(sample_vector)

print(prediction)

[0]
